This is the Hausa section of the COS 760 final project

Specific steps and todo:   

- Language-specific filtering
- Data Cleaning and text standarization
- Multilabel label preparation
- Fixing data imbalancements
- Tokenization
- Baseline model
- Preperation for augmentation

**Language specific filtering**

In [ ]:
!pip install pandas numpy scikit-learn transformers datasets torch evaluate
!pip install sentencepiece sacremoses
!pip install accelerate -U
!pip install tqdm

In [2]:
from datasets import load_dataset, Dataset, concatenate_datasets

import pandas as pd
import numpy as np
import re
import html
import unicodedata
import random
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, AutoTokenizer, MarianMTModel, MarianTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import f1_score, accuracy_score
import evaluate
import torch
import os
from tqdm.notebook import tqdm
import requests
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

dataset = load_dataset("brighter-dataset/BRIGHTER-emotion-categories", "hau")
print(dataset)
print(dataset['train'][0])
df = pd.DataFrame(dataset['train'])
print(df.describe())

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'emotions'],
        num_rows: 2145
    })
    dev: Dataset({
        features: ['id', 'text', 'anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'emotions'],
        num_rows: 712
    })
    test: Dataset({
        features: ['id', 'text', 'anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'emotions'],
        num_rows: 2160
    })
})
{'id': 'hau_train_track_a_00001', 'text': "kotu ta yi hukunci kan shari'ar zaben dan majalisar pdp, ta yi hukuncin bazata", 'anger': 0, 'disgust': 0, 'fear': 0, 'joy': 0, 'sadness': 0, 'surprise': 1, 'emotions': ['surprise']}
             anger      disgust         fear          joy      sadness  \
count  2145.000000  2145.000000  2145.000000  2145.000000  2145.000000   
mean      0.190210     0.153380     0.152448     0.149184     0.301632   
std       0.392558     0.360437     0.359538     0.356353     0.459073   


**Data cleaning and text standardization**

In [3]:
def clean_text(text):
    #UTF-8
    if isinstance(text, bytes):
        text = text.decode("utf-8", errors="ignore")
    text = unicodedata.normalize("NFKC", text)

    text = text.lower()

    #Remove any HTML
    text = re.sub(r"<.?>", " ", text)
    text = html.unescape(text)

    #Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [4]:
dataset = dataset.map(lambda x: {"text": clean_text(x["text"])})
print(dataset['train'][0]['text'])

kotu ta yi hukunci kan shari'ar zaben dan majalisar pdp, ta yi hukuncin bazata


**Multilabel label preparation**

In [5]:
base_emotions = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
emotion_cols  = base_emotions + ["neutral"]

***Creating target vector***

In [6]:
df["target"] = df[base_emotions].values.tolist()
print(df[["text", "target"]].head())

                                                text              target
0  kotu ta yi hukunci kan shari'ar zaben dan maja...  [0, 0, 0, 0, 0, 1]
1  toh fah inji 'yan magana suka ce """"ana wata ...  [0, 0, 0, 0, 0, 1]
2  bincike ya nuna yan najeriya sun fi damuwa da ...  [0, 0, 1, 0, 1, 0]
3  kwamishina ya musanta rahoton masari ya cire k...  [0, 0, 0, 0, 0, 0]
4  innalillahi wa inna ilaihir raji'un: allah ya ...  [0, 0, 0, 0, 1, 0]


In [7]:
#Ensures that the emotions list matches the binary columns
def check_consistency(row):
    binary_labels = [col for col in base_emotions if row[col] == 1]
    list_labels = row["emotions"] if row["emotions"] else []

    return set(binary_labels) == set(list_labels)

In [8]:
df["consistent"] = df.apply(check_consistency, axis=1)
inconsistent_rows = df[~df["consistent"]]
print(f"Inconsistent rows: {len(inconsistent_rows)}")

Inconsistent rows: 0


**Fixing data imbalancement**

- Rebalancing train and test split

In [9]:
raw_train_df = pd.DataFrame(dataset["train"])
raw_train_df["text"] = [clean_text(t) for t in raw_train_df["text"]]

raw_train_df["neutral"] = (raw_train_df[base_emotions].fillna(0).sum(axis=1) == 0).astype(int)

test_df_full = pd.DataFrame(dataset["test"])
test_df_full["text"] = [clean_text(t) for t in test_df_full["text"]]

moved_to_train = test_df_full.sample(n=800, random_state=42)
remaining_test = test_df_full.drop(moved_to_train.index).reset_index(drop=True)

moved_to_train = moved_to_train.copy()
moved_to_train["neutral"] = (moved_to_train[base_emotions].fillna(0).sum(axis=1) == 0).astype(int)

raw_train_df = pd.concat([raw_train_df, moved_to_train], ignore_index=True)

print(f"New training size : {len(raw_train_df)}")
print(f"New test size     : {len(remaining_test)}")


New training size : 2945
New test size     : 1360


***Tokenization***

In [10]:
def preprocess_dataframe(df, emotion_cols):
    df = df.copy()

    df[emotion_cols[:-1]] = df[emotion_cols[:-1]].fillna(0).astype(int)

    df["neutral"] = (df[emotion_cols[:-1]].sum(axis=1) == 0).astype(int)
    df["labels"] = df[emotion_cols].values.tolist()

    return df

In [11]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-large")

In [12]:
def tokenize(data):
    return tokenizer(
        data["text"],
        padding="max_length",
        truncation=True,
        max_length=45
    )

In [13]:
def convert_labels(data):
    tensor = torch.tensor(data["labels"], dtype=torch.float32)
    n_labels = tensor.shape[0] if tensor.ndim == 1 else tensor.shape[1]
    assert n_labels == len(emotion_cols), f"Label shape mismatch: {tensor.shape}"

    data["labels"] = tensor.tolist()
    return data

In [14]:
def make_torch_dataset(df):
    ds = Dataset.from_pandas(df)
    ds = ds.map(tokenize, batched=True).map(convert_labels)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

In [15]:
train_df = preprocess_dataframe(raw_train_df.copy(), emotion_cols)
val_df   = preprocess_dataframe(
    pd.DataFrame(dataset["dev"]),
    emotion_cols,
)
test_df  = preprocess_dataframe(
    remaining_test.copy(),
    emotion_cols,
)

train_dataset = make_torch_dataset(train_df)
val_dataset   = make_torch_dataset(val_df)
test_dataset  = make_torch_dataset(test_df)

print(train_dataset)

Map:   0%|          | 0/2945 [00:00<?, ? examples/s]

Map:   0%|          | 0/2945 [00:00<?, ? examples/s]

Map:   0%|          | 0/712 [00:00<?, ? examples/s]

Map:   0%|          | 0/712 [00:00<?, ? examples/s]

Map:   0%|          | 0/1360 [00:00<?, ? examples/s]

Map:   0%|          | 0/1360 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'text', 'anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'emotions', 'neutral', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 2945
})


In [16]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.BCEWithLogitsLoss()
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [17]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = (torch.sigmoid(torch.tensor(predictions)) > 0.3).numpy()

    # Calculate metrics
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)
    f1_micro = f1_score(labels, predictions, average='micro', zero_division=0)
    f1_weighted = f1_score(labels, predictions, average='weighted', zero_division=0)

    return {
        'f1_macro': f1_macro,
        'f1_micro': f1_micro,
        'f1_weighted': f1_weighted
    }

In [18]:
def create_training_args(output_dir, lr=1e-5, epochs=5):
    return TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=50,
        dataloader_pin_memory=False,
        bf16=False,
        fp16=False
    )

In [19]:
def train_and_evaluate(model_name, train_ds, val_ds, test_ds, output_dir, lr=1e-5, epochs=5):
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(emotion_cols),
        problem_type="multi_label_classification",
    ).float()
    args = create_training_args(output_dir, lr=lr, epochs=epochs)
    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()

    trainer.callback_handler.callbacks = [
        cb for cb in trainer.callback_handler.callbacks
        if not cb.__class__.__name__ == "NotebookProgressCallback"
    ]
    results = trainer.evaluate(test_ds)
    return results


In [20]:
XLM_R_MODEL   = "xlm-roberta-large"
AFROXLMR_MODEL = "Davlan/afro-xlmr-large"

In [25]:
print("\n" + "="*60)
print("CONDITION A – Baseline: XLM-RoBERTa-large")
print("="*60)
xlmr_baseline_results = train_and_evaluate(
    XLM_R_MODEL,
    train_dataset, val_dataset, test_dataset,
    output_dir="./results_xlmr_baseline",
    lr=1e-5
)
print(f"Test F1 Macro: {xlmr_baseline_results['eval_f1_macro']:.4f}")
print(f"Test F1 Micro: {xlmr_baseline_results['eval_f1_micro']:.4f}")
print(f"Test F1 Weighted: {xlmr_baseline_results['eval_f1_weighted']:.4f}")

print("\n" + "="*60)
print("CONDITION A – Baseline: AfroXLMR-large")
print("="*60)
afroxlmr_baseline_results = train_and_evaluate(
    AFROXLMR_MODEL,
    train_dataset, val_dataset, test_dataset,
    output_dir="./results_afroxlmr_baseline",
    lr=1e-5
)
print(f"Test F1 Macro: {afroxlmr_baseline_results['eval_f1_macro']:.4f}")
print(f"Test F1 Micro: {afroxlmr_baseline_results['eval_f1_micro']:.4f}")
print(f"Test F1 Weighted: {afroxlmr_baseline_results['eval_f1_weighted']:.4f}")


CONDITION A – Baseline: XLM-RoBERTa-large


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,F1 Weighted
1,0.441620,0.421362,0.229736,0.380597,0.278953
2,0.398329,0.362807,0.381042,0.480982,0.417402
3,0.350591,0.334069,0.548180,0.577480,0.566745
4,0.330646,0.323813,0.554744,0.596983,0.579078
5,0.296898,0.312045,0.584967,0.607759,0.601507


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Test F1 Macro: 0.6645
Test F1 Micro: 0.6757
Test F1 Weighted: 0.6731


**Building augmented training set using back-translation**

Two pilot languages are used:
- English
- French

In [21]:
model_cache: dict = {}

def load_translation_model(model_name: str):
    if model_name not in model_cache:
        print(f"  Loading translation model: {model_name}")
        tok = MarianTokenizer.from_pretrained(model_name)
        mdl = MarianMTModel.from_pretrained(model_name)
        mdl.eval()
        if torch.cuda.is_available():
            mdl = mdl.cuda()
        model_cache[model_name] = (tok, mdl)
    return model_cache[model_name]

In [22]:
def translate_batch(texts: list[str], model_name: str, batch_size: int = 32) -> list[str]:
    tok, mdl = load_translation_model(model_name)
    device    = next(mdl.parameters()).device
    results   = []

    for i in tqdm(range(0, len(texts), batch_size),
                  desc=f"Translating [{model_name.split('/')[-1]}]",
                  leave=False):
        batch = texts[i : i + batch_size]
        encoded = tok(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128,
        ).to(device)

        with torch.no_grad():
            translated_ids = mdl.generate(
                **encoded,
                num_beams=1,
                max_length=128,
            )

        decoded = tok.batch_decode(translated_ids, skip_special_tokens=True)
        results.extend(decoded)

    return results

In [23]:
PIVOT_MODELS = {
    "english": {
        "forward":  "Helsinki-NLP/opus-mt-ha-en",
        "backward": "Helsinki-NLP/opus-mt-en-ha",
    },
    "french": {
        "forward":  "Helsinki-NLP/opus-mt-ha-fr",
        "backward": "Helsinki-NLP/opus-mt-fr-ha",
    },
}

In [24]:
def back_translate(
    texts: list[str],
    pivot: str = "english",
    batch_size: int = 32,
) -> list[str]:
    pivot = pivot.lower()
    if pivot not in PIVOT_MODELS:
        raise ValueError(f"Unknown pivot '{pivot}'. Choose from {list(PIVOT_MODELS)}.")

    fwd = PIVOT_MODELS[pivot]["forward"]
    bwd = PIVOT_MODELS[pivot]["backward"]

    print(f"\n[Back-translation] Hausa → {pivot.capitalize()} …")
    intermediate = translate_batch(texts, fwd, batch_size=batch_size)

    print(f"[Back-translation] {pivot.capitalize()} → Hausa …")
    back          = translate_batch(intermediate, bwd, batch_size=batch_size)

    return back

In [25]:
def build_augmented_df(
    original_df:   pd.DataFrame,
    augmented_texts: list[str],
    pivot_label:   str,
    emotion_cols:  list[str],
) -> pd.DataFrame:
    aug_df = original_df.copy().reset_index(drop=True)
    aug_df["text"] = augmented_texts
    aug_df["augmentation"] = pivot_label

    # Drop rows where translation is identical to the original
    original_texts_reset = original_df["text"].reset_index(drop=True)
    identical_mask = aug_df["text"] == original_texts_reset
    n_identical = identical_mask.sum()
    if n_identical:
        print(f"  [{pivot_label}] Dropping {n_identical} unchanged translations.")
    aug_df = aug_df[~identical_mask].reset_index(drop=True)

    # Recompute labels to ensure consistency after possible column drift
    aug_df = preprocess_dataframe(aug_df, emotion_cols)

    return aug_df

In [26]:
raw_train_sample = raw_train_df.sample(
    n=int(len(raw_train_df) / 4), random_state=42
).reset_index(drop=True)
sampled_texts = raw_train_sample["text"].tolist()

print("Running back-translation via English …")
bt_english_texts = back_translate(sampled_texts, pivot="english", batch_size=32)

print("\nRunning back-translation via French …")
bt_french_texts   = back_translate(sampled_texts, pivot="french",   batch_size=32)

aug_english_df = build_augmented_df(raw_train_sample, bt_english_texts, "bt_english", emotion_cols)
aug_french_df   = build_augmented_df(raw_train_sample, bt_french_texts,   "bt_french",   emotion_cols)

print(f"\nOriginal training rows   : {len(raw_train_df)}")
print(f"Augmented (English BT)   : {len(aug_english_df)}")
print(f"Augmented (French   BT)   : {len(aug_french_df)}")

Running back-translation via English …

[Back-translation] Hausa → English …
  Loading translation model: Helsinki-NLP/opus-mt-ha-en


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Translating [opus-mt-ha-en]:   0%|          | 0/23 [00:00<?, ?it/s]

[Back-translation] English → Hausa …
  Loading translation model: Helsinki-NLP/opus-mt-en-ha


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

...

Translating [opus-mt-ha-fr]:   0%|          | 0/23 [00:00<?, ?it/s]

[Back-translation] French → Hausa …
  Loading translation model: Helsinki-NLP/opus-mt-fr-ha


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Translating [opus-mt-fr-ha]:   0%|          | 0/23 [00:00<?, ?it/s]


Original training rows   : 2945
Augmented (English BT)   : 736
Augmented (French   BT)   : 736


In [27]:
original_train_df = preprocess_dataframe(raw_train_df.copy(), emotion_cols)
original_train_df["augmentation"] = "original"

bt_train_df = pd.concat(
    [original_train_df, aug_english_df, aug_french_df],
    ignore_index=True,
).sample(frac=1, random_state=42)

print(f"\nCondition B training set size : {len(bt_train_df)}")
print(bt_train_df["augmentation"].value_counts())
bt_train_dataset = make_torch_dataset(bt_train_df)


Condition B training set size : 4417
augmentation
original      2945
bt_french      736
bt_english     736
Name: count, dtype: int64


Map:   0%|          | 0/4417 [00:00<?, ? examples/s]

Map:   0%|          | 0/4417 [00:00<?, ? examples/s]

In [35]:
print("\n" + "="*60)
print("CONDITION B – Back-translation: XLM-RoBERTa-large")
print("="*60)
xlmr_bt_results = train_and_evaluate(
    XLM_R_MODEL,
    bt_train_dataset, val_dataset, test_dataset,
    output_dir="./results_xlmr_bt",
    lr=1e-5,
)
print(f"Test F1 Macro: {xlmr_bt_results['eval_f1_macro']:.4f}")
print(f"Test F1 Micro: {xlmr_bt_results['eval_f1_micro']:.4f}")
print(f"Test F1 Weighted: {xlmr_bt_results['eval_f1_weighted']:.4f}")

print("\n" + "="*60)
print("CONDITION B – Back-translation: AfroXLMR-large")
print("="*60)
afroxlmr_bt_results = train_and_evaluate(
    AFROXLMR_MODEL,
    bt_train_dataset, val_dataset, test_dataset,
    output_dir="./results_afroxlmr_bt",
    lr=1e-5,
)
print(f"Test F1 Macro: {afroxlmr_bt_results['eval_f1_macro']:.4f}")
print(f"Test F1 Micro: {afroxlmr_bt_results['eval_f1_micro']:.4f}")
print(f"Test F1 Weighted: {afroxlmr_bt_results['eval_f1_weighted']:.4f}")



CONDITION B – Back-translation: XLM-RoBERTa-large


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,F1 Weighted
1,0.428313,0.386366,0.306288,0.434663,0.344044
2,0.381876,0.334705,0.530982,0.586169,0.555533
3,0.342634,0.304407,0.595954,0.622363,0.611385
4,0.311163,0.290865,0.605899,0.625393,0.621147
5,0.303612,0.289152,0.614630,0.640509,0.630825


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Test F1 Macro: 0.5900
Test F1 Micro: 0.6090
Test F1 Weighted: 0.6021


**Paraphrasing**

Uses Ollama (local) with gemma4:e4b to rewrite Afrikaans sentences while preserving their emotion labels.  The prompt explicitly names the active emotions so the model knows what it must not change.

In [28]:
OLLAMA_URL    = "http://localhost:11434/api/chat"
OLLAMA_MODEL  = "gemma4:e4b"

In [29]:
def emotion_label_string(row: pd.Series, emotion_cols: list[str]) -> str:
    active = [e for e in emotion_cols if row.get(e, 0) == 1]
    return ", ".join(active) if active else "neutral"

In [30]:
def build_paraphrase_prompt(text: str, emotions: str) -> str:
    return (
        f"Sake rubuta jimlar Hausa mai zuwa ta wata sabuwar hanya. "
        f"Dole ne jimlar ta ci gaba da nuna irin wannan motsin rai: {emotions}. "
        f"Ka bayar da jimlar da aka sake rubutawa kawai, ba tare da wani bayani ba.\n\n"
        f"Jimlar asali: {text}\n"
        f"Jimlar da aka sake rubutawa:"
    )

In [31]:
def paraphrase_text(
    text: str,
    emotions: str,
    retries: int = 3,
    timeout: int = 60,
) -> str | None:
    prompt = build_paraphrase_prompt(text, emotions)

    for attempt in range(retries):
        try:
            resp = requests.post(
                OLLAMA_URL,
                json={
                    "model": OLLAMA_MODEL,
                    "think": False,
                    "stream": False,
                    "messages": [
                        {"role": "user", "content": prompt}
                    ],
                    "options": {
                        "temperature": 0.7,
                        "top_p": 0.9,
                        "num_predict": 512,
                    },
                },
                timeout=timeout,
            )
            resp.raise_for_status()
            data = resp.json()

            result = data.get("message", {}).get("content", "").strip()

            for prefix in [
                "Jimlar da aka sake rubutawa:",
                "Amsa:",
                "**Jimlar da aka sake rubutawa:**"
            ]:
                if result.lower().startswith(prefix.lower()):
                    result = result[len(prefix):].strip()

            result = result.splitlines()[0].strip() if result else ""

            if result:
                return result

        except requests.exceptions.RequestException as e:
            print("ERROR:", e)

            if 'resp' in locals():
                print(resp.text)

            wait = 2 ** attempt
            print(f"  [Ollama] Attempt {attempt+1} failed: {e}. Retrying in {wait}s …")
            time.sleep(wait)

    return None

In [32]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def paraphrase_dataframe(df: pd.DataFrame, emotion_cols: list[str], max_workers: int = 4) -> list[str]:
    results   = [None] * len(df)
    fallbacks = 0

    def _worker(idx_row):
        idx, row = idx_row
        emotions   = emotion_label_string(row, emotion_cols)
        paraphrase = paraphrase_text(row["text"], emotions)
        return idx, paraphrase if paraphrase is not None else row["text"]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_worker, (i, row)): i
                   for i, row in df.iterrows()}
        for future in tqdm(as_completed(futures), total=len(df), desc="Paraphrasing [Ollama]"):
            idx, text = future.result()
            results[idx - df.index[0]] = text

    fallbacks = sum(1 for i, (_, row) in enumerate(df.iterrows()) if results[i] == row["text"])
    if fallbacks:
        print(f"  [Paraphrase] {fallbacks}/{len(df)} rows used original text as fallback.")
    return results

In [33]:
para_sample = raw_train_df.sample(
    n=int(len(raw_train_df) / 2), random_state=99
).reset_index(drop=True)

paraphrase_texts = paraphrase_dataframe(para_sample, emotion_cols)

aug_paraphrase_df = build_augmented_df(
    para_sample,
    paraphrase_texts,
    "paraphrase",
    emotion_cols,
)
print(f"Paraphrase augmented rows : {len(aug_paraphrase_df)}")

Paraphrasing [Ollama]:   0%|          | 0/1472 [00:00<?, ?it/s]

Paraphrase augmented rows : 1472


In [34]:
para_only_train_df = pd.concat(
    [original_train_df, aug_paraphrase_df],
    ignore_index=True,
).sample(frac=1, random_state=42)

print(f"\nCondition C training set size : {len(para_only_train_df)}")
print(para_only_train_df["augmentation"].value_counts())

para_only_train_dataset = make_torch_dataset(para_only_train_df)


print("\n" + "="*60)
print("CONDITION C – Paraphrase-only: XLM-RoBERTa-large")
print("="*60)
xlmr_para_results = train_and_evaluate(
    XLM_R_MODEL,
    para_only_train_dataset, val_dataset, test_dataset,
    output_dir="./results_xlmr_para",
    lr=1e-5,
)

print("\n" + "="*60)
print("CONDITION C – Paraphrase-only: AfroXLMR-large")
print("="*60)
afroxlmr_para_results = train_and_evaluate(
    AFROXLMR_MODEL,
    para_only_train_dataset, val_dataset, test_dataset,
    output_dir="./results_afroxlmr_para",
    lr=1e-5,
)


Condition C training set size : 4417
augmentation
original      2945
paraphrase    1472
Name: count, dtype: int64


Map:   0%|          | 0/4417 [00:00<?, ? examples/s]

Map:   0%|          | 0/4417 [00:00<?, ? examples/s]


CONDITION C – Paraphrase-only: XLM-RoBERTa-large


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

In [35]:
combined_train_df = pd.concat(
    [original_train_df, aug_english_df, aug_french_df, aug_paraphrase_df],
    ignore_index=True,
).sample(frac=1, random_state=42)

print(f"\nCondition D training set size : {len(combined_train_df)}")
print(combined_train_df["augmentation"].value_counts())

combined_train_dataset = make_torch_dataset(combined_train_df)


print("\n" + "="*60)
print("CONDITION D – Combined: XLM-RoBERTa-large")
print("="*60)
xlmr_combined_results = train_and_evaluate(
    XLM_R_MODEL,
    combined_train_dataset, val_dataset, test_dataset,
    output_dir="./results_xlmr_combined",
    lr=1e-5,
)

print("\n" + "="*60)
print("CONDITION D – Combined: AfroXLMR-large")
print("="*60)
afroxlmr_combined_results = train_and_evaluate(
    AFROXLMR_MODEL,
    combined_train_dataset, val_dataset, test_dataset,
    output_dir="./results_afroxlmr_combined",
    lr=1e-5,
)


Condition D training set size : 5889
augmentation
original      2945
paraphrase    1472
bt_french      736
bt_english     736
Name: count, dtype: int64


Map:   0%|          | 0/5889 [00:00<?, ? examples/s]

Map:   0%|          | 0/5889 [00:00<?, ? examples/s]


CONDITION D – Combined: XLM-RoBERTa-large


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

In [40]:
comparison_rows = [
    #("A – Baseline",          "XLM-RoBERTa-large",  xlmr_baseline_results),
    #("A – Baseline",          "AfroXLMR-large",      afroxlmr_baseline_results),
    #("B – Back-translation",  "XLM-RoBERTa-large",  xlmr_bt_results),
    #("B – Back-translation",  "AfroXLMR-large",      afroxlmr_bt_results),
    ("C – Paraphrase-only",   "XLM-RoBERTa-large",  xlmr_para_results),
    ("C – Paraphrase-only",   "AfroXLMR-large",      afroxlmr_para_results),
    ("D – Combined",          "XLM-RoBERTa-large",  xlmr_combined_results),
    ("D – Combined",          "AfroXLMR-large",      afroxlmr_combined_results),
]

col_w = (25, 22, 10, 10, 12)
header = (
    f"{'Condition':<{col_w[0]}}"
    f"{'Model':<{col_w[1]}}"
    f"{'F1 Macro':>{col_w[2]}}"
    f"{'F1 Micro':>{col_w[3]}}"
    f"{'F1 Weighted':>{col_w[4]}}"
)
sep = "-" * sum(col_w)

print("\n" + "="*sum(col_w))
print("HAUSA – FULL RESULTS SUMMARY")
print("="*sum(col_w))
print(header)
print(sep)

for cond, model_tag, res in comparison_rows:
    print(
        f"{cond:<{col_w[0]}}"
        f"{model_tag:<{col_w[1]}}"
        f"{res['eval_f1_macro']:>{col_w[2]}.4f}"
        f"{res['eval_f1_micro']:>{col_w[3]}.4f}"
        f"{res['eval_f1_weighted']:>{col_w[4]}.4f}"
    )

print(sep)

#xlmr_base_macro = xlmr_baseline_results['eval_f1_macro']

# print("\nΔ F1 Macro relative to XLM-RoBERTa Baseline:")
# print(sep)
# for cond, model_tag, res in comparison_rows:
#     delta = res['eval_f1_macro'] - xlmr_base_macro
#     print(
#         f"{cond:<{col_w[0]}}"
#         f"{model_tag:<{col_w[1]}}"
#         f"  {delta:>+.4f}"
#     )
# print(sep)


HAUSA – FULL RESULTS SUMMARY
Condition                Model                   F1 Macro  F1 Micro F1 Weighted
-------------------------------------------------------------------------------
C – Paraphrase-only      XLM-RoBERTa-large         0.6550    0.6623      0.6652
C – Paraphrase-only      AfroXLMR-large            0.7299    0.7352      0.7360
D – Combined             XLM-RoBERTa-large         0.6404    0.6498      0.6508
D – Combined             AfroXLMR-large            0.6811    0.6843      0.6868
-------------------------------------------------------------------------------
